https://coinmarketcap.com/rankings/exchanges/

In [1]:
# 1. Imports
from pathlib import Path
import time
import configparser
from datetime import datetime

import ccxt
import pandas as pd
import requests
from tqdm.auto import tqdm

In [ ]:
# 2. Configuration
START_DATE = "2026-01-01"
END_DATE = "2026-01-31" # inclusive
VS_CURRENCY = "usd"

TOP_N = 100
TOP_EXCHANGES_N = 20

UPDATE_TOP_EXCHANGE_COUNT = True

DATA_DIR = Path("../../data")
RAW_DIR = DATA_DIR / "raw"
FETCHED_DIR = RAW_DIR / "fetched"

ASSETS_PATH = FETCHED_DIR / "assets.parquet"
ASSET_DAILY_PATH = FETCHED_DIR / "asset_daily.parquet"

COINGECKO_BASE_URL = "https://api.coingecko.com/api/v3"
REQUEST_SLEEP_SECONDS = 2.05
REQUEST_TIMEOUT_SECONDS = 30
USER_AGENT = "TDA-Data-Aquisition"

USD_LIKE_QUOTES = ["USDT", "USDC", "USD"]

config = configparser.ConfigParser()
config.read("../../secret.cfg")

COINGECKO_API_KEY = config.get("COINGECKO", "DEMO_API_KEY")

if not COINGECKO_API_KEY:
    raise ValueError("CoinGecko DEMO_API_KEY not found in secret.cfg")

ASSET_COLUMNS = [
    "asset_id",
    "coingecko_id",
    "symbol",
    "name",
    "circulating_supply",
    "total_supply",
    "max_supply",
    "top_exchange_count",
]

ASSET_DAILY_COLUMNS = [
    "date",
    "asset_id",
    "price_usd",
    "market_cap_usd",
    "global_volume_24h_usd",
    "binance_volume_24h_usd",
]

In [3]:
# 3. Request helper
def coingecko_get(endpoint: str, params: dict | None = None, max_retries: int = 5) -> dict | list:
    url = f"{COINGECKO_BASE_URL}{endpoint}"
    headers = {
        "User-Agent": USER_AGENT,
        "x-cg-demo-api-key": COINGECKO_API_KEY,
    }
    params = dict(params or {})
    last_error = None

    for attempt in range(max_retries):
        response = requests.get(url, params=params, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)

        if response.status_code == 429:
            wait_seconds = min(90, 5 * (attempt + 1))
            print(f"429 for {endpoint} | attempt {attempt + 1}/{max_retries} | sleeping {wait_seconds}s")
            time.sleep(wait_seconds)
            last_error = RuntimeError(f"429 rate limit for {url}")
            continue

        response.raise_for_status()
        time.sleep(REQUEST_SLEEP_SECONDS)
        return response.json()

    raise last_error or RuntimeError(f"Request failed for {url}")

In [4]:
# 4. API-test
print(coingecko_get("/ping"))

{'gecko_says': '(V3) To the Moon!'}


In [5]:
# 5. Storage and validation helpers
def ensure_output_dirs() -> None:
    FETCHED_DIR.mkdir(parents=True, exist_ok=True)


def enforce_columns(df: pd.DataFrame, expected_columns: list[str]) -> pd.DataFrame:
    missing = [c for c in expected_columns if c not in df.columns]
    extra = [c for c in df.columns if c not in expected_columns]

    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if extra:
        raise ValueError(f"Unexpected columns: {extra}")

    return df[expected_columns].copy()


def ensure_asset_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "top_exchange_count" not in df.columns:
        df["top_exchange_count"] = pd.NA

    return enforce_columns(df, ASSET_COLUMNS)


def validate_date_range(start_date: str, end_date: str) -> tuple[pd.Timestamp, pd.Timestamp]:
    start_dt = pd.Timestamp(start_date)
    end_dt = pd.Timestamp(end_date)

    if start_dt > end_dt:
        raise ValueError(f"START_DATE must be on or before END_DATE. Got {start_date} > {end_date}.")

    today_utc = pd.Timestamp.now(tz="UTC").normalize()
    last_finished_day = today_utc - pd.Timedelta(days=1)

    if start_dt.tz_localize("UTC") > last_finished_day:
        raise ValueError(f"START_DATE must be a finished UTC day. Latest allowed date is {last_finished_day.date()}.")

    if end_dt.tz_localize("UTC") > last_finished_day:
        raise ValueError(f"END_DATE must be a finished UTC day. Latest allowed date is {last_finished_day.date()}.")

    return start_dt, end_dt


def make_date_list(start_date: str, end_date: str) -> list[str]:
    start_dt, end_dt = validate_date_range(start_date, end_date)
    return [d.strftime("%Y-%m-%d") for d in pd.date_range(start=start_dt, end=end_dt, freq="D")]


def load_existing_assets() -> pd.DataFrame:
    if ASSETS_PATH.exists():
        df = pd.read_parquet(ASSETS_PATH)
        return ensure_asset_columns(df)
    return pd.DataFrame(columns=ASSET_COLUMNS)


def load_existing_asset_daily() -> pd.DataFrame:
    if ASSET_DAILY_PATH.exists():
        df = pd.read_parquet(ASSET_DAILY_PATH)
        return enforce_columns(df, ASSET_DAILY_COLUMNS)
    return pd.DataFrame(columns=ASSET_DAILY_COLUMNS)

In [6]:
# 6. Progress bar helpers
def make_text_progress_bar(total: int, desc: str) -> tqdm:
    return tqdm(
        total=total,
        desc=desc,
        leave=True,
        dynamic_ncols=True,
        mininterval=0.5,
        colour="green",
    )

In [7]:
# 7. Binance helpers
def make_binance_exchange():
    exchange = ccxt.binance(
        {
            "enableRateLimit": True,
            "timeout": 60000,
        }
    )
    exchange.load_markets()
    return exchange


def find_binance_usd_like_symbol(exchange, base_symbol: str) -> str | None:
    base_symbol = str(base_symbol).upper().strip()

    for quote in USD_LIKE_QUOTES:
        candidate = f"{base_symbol}/{quote}"
        if candidate in exchange.markets:
            market = exchange.markets[candidate]
            if market.get("spot") and market.get("active", True):
                return candidate

    return None


def fetch_binance_quote_volume_for_date(exchange, market_symbol: str, date_str: str) -> float | None:
    market = exchange.market(market_symbol)

    start_ts = int(exchange.parse8601(f"{date_str}T00:00:00Z"))
    end_ts = int(start_ts + 24 * 60 * 60 * 1000)

    rows = exchange.publicGetKlines(
        {
            "symbol": str(market["id"]),
            "interval": "1d",
            "startTime": start_ts,
            "endTime": end_ts,
            "limit": 1,
        }
    )

    if not rows:
        return None

    row = rows[0]

    candle_date = pd.to_datetime(int(row[0]), unit="ms", utc=True).strftime("%Y-%m-%d")
    if candle_date != date_str:
        return None

    return float(row[7])


def fetch_binance_quote_volume_for_date_with_retry(
    exchange,
    market_symbol: str,
    date_str: str,
    max_retries: int = 4,
    base_sleep_seconds: float = 5.0,
) -> float | None:
    last_error = None

    for attempt in range(max_retries):
        try:
            return fetch_binance_quote_volume_for_date(
                exchange=exchange,
                market_symbol=market_symbol,
                date_str=date_str,
            )
        except (ccxt.NetworkError, ccxt.RequestTimeout, ccxt.ExchangeNotAvailable) as e:
            last_error = e
            wait_seconds = base_sleep_seconds * (attempt + 1)
            print(
                f"Temporary Binance error for {market_symbol} on {date_str} | "
                f"attempt {attempt + 1}/{max_retries} | sleeping {wait_seconds}s | error={repr(e)}"
            )
            time.sleep(wait_seconds)

    raise last_error

In [8]:
# 8. CoinGecko fetch helpers
def to_coingecko_history_date(date_str: str) -> str:
    dt = datetime.strptime(date_str, "%Y-%m-%d")
    return dt.strftime("%d-%m-%Y")


def fetch_top_n_universe_for_date(date_str: str, top_n: int) -> pd.DataFrame:
    data = coingecko_get(
        "/coins/markets",
        params={
            "vs_currency": VS_CURRENCY,
            "order": "market_cap_desc",
            "per_page": top_n,
            "page": 1,
            "sparkline": "false",
            "price_change_percentage": "24h",
            "date": to_coingecko_history_date(date_str),
        },
    )

    df = pd.DataFrame(data)

    required_cols = [
        "id",
        "symbol",
        "name",
        "current_price",
        "market_cap",
        "total_volume",
        "circulating_supply",
        "total_supply",
        "max_supply",
    ]

    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"CoinGecko response missing required fields: {missing_cols}")

    return df[required_cols].copy()


def fetch_asset_history_row(coingecko_id: str, date_str: str) -> dict:
    history = coingecko_get(
        f"/coins/{coingecko_id}/history",
        params={
            "date": to_coingecko_history_date(date_str),
            "localization": "false",
        },
    )

    market_data = history.get("market_data") or {}
    current_price = market_data.get("current_price") or {}
    market_cap = market_data.get("market_cap") or {}
    total_volume = market_data.get("total_volume") or {}

    return {
        "date": date_str,
        "coingecko_id": coingecko_id,
        "price_usd": current_price.get("usd"),
        "market_cap_usd": market_cap.get("usd"),
        "global_volume_24h_usd": total_volume.get("usd"),
    }

In [9]:
# 9. Asset table helpers
def merge_assets_cumulative(existing_assets: pd.DataFrame, new_asset_core_df: pd.DataFrame) -> pd.DataFrame:
    existing_assets = ensure_asset_columns(existing_assets.copy())
    new_asset_core_df = new_asset_core_df.copy()

    if existing_assets.empty:
        new_asset_core_df = new_asset_core_df.reset_index(drop=True)
        new_asset_core_df.insert(0, "asset_id", range(1, len(new_asset_core_df) + 1))
        new_asset_core_df["top_exchange_count"] = pd.NA
        return ensure_asset_columns(new_asset_core_df)

    new_asset_core_df = new_asset_core_df[
        ["coingecko_id", "symbol", "name", "circulating_supply", "total_supply", "max_supply"]
    ].copy()

    existing_ids_df = existing_assets[["asset_id", "coingecko_id", "top_exchange_count"]].copy()

    new_with_ids = new_asset_core_df.merge(
        existing_ids_df,
        on="coingecko_id",
        how="left",
    )

    next_asset_id = int(existing_assets["asset_id"].max()) + 1
    missing_mask = new_with_ids["asset_id"].isna()
    missing_count = int(missing_mask.sum())

    if missing_count > 0:
        new_with_ids.loc[missing_mask, "asset_id"] = range(next_asset_id, next_asset_id + missing_count)

    new_with_ids["asset_id"] = new_with_ids["asset_id"].astype(int)

    new_with_ids = new_with_ids[
        [
            "asset_id",
            "coingecko_id",
            "symbol",
            "name",
            "circulating_supply",
            "total_supply",
            "max_supply",
            "top_exchange_count",
        ]
    ].copy()

    existing_indexed = existing_assets.set_index("coingecko_id")
    new_indexed = new_with_ids.set_index("coingecko_id")

    cols_to_update = [
        "asset_id",
        "symbol",
        "name",
        "circulating_supply",
        "total_supply",
        "max_supply",
    ]
    existing_indexed.update(new_indexed[cols_to_update])

    new_only = new_indexed.loc[~new_indexed.index.isin(existing_indexed.index)].copy()

    combined = pd.concat([existing_indexed, new_only], axis=0).reset_index()

    combined = combined[
        [
            "asset_id",
            "coingecko_id",
            "symbol",
            "name",
            "circulating_supply",
            "total_supply",
            "max_supply",
            "top_exchange_count",
        ]
    ].copy()

    combined["asset_id"] = combined["asset_id"].astype(int)
    combined = combined.sort_values("asset_id").reset_index(drop=True)

    return ensure_asset_columns(combined)


def append_only_new_asset_daily(existing_df: pd.DataFrame, new_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    existing_df = existing_df.copy()
    new_df = new_df.copy()

    if existing_df.empty:
        combined = new_df.sort_values(["date", "asset_id"]).reset_index(drop=True)
        return combined, new_df.copy()

    existing_keys = set(zip(existing_df["date"].astype(str), existing_df["asset_id"].astype(int)))
    is_new_mask = new_df.apply(lambda r: (str(r["date"]), int(r["asset_id"])) not in existing_keys, axis=1)

    appended_df = new_df.loc[is_new_mask].copy()

    combined = pd.concat([existing_df, appended_df], ignore_index=True)
    combined = combined.sort_values(["date", "asset_id"]).reset_index(drop=True)

    return combined, appended_df

In [10]:
# 10. Top-exchange count helpers
def fetch_top_exchange_names_from_cmc(top_n: int) -> pd.DataFrame:
    tables = pd.read_html("https://coinmarketcap.com/rankings/exchanges/")
    if not tables:
        raise ValueError("Could not read exchange table from CoinMarketCap rankings page.")

    cmc_df = tables[0].copy()

    name_col = None
    for candidate in cmc_df.columns:
        candidate_str = str(candidate).strip().lower()
        if "exchange" in candidate_str or candidate_str == "name":
            name_col = candidate
            break

    if name_col is None:
        raise ValueError(f"Could not find exchange-name column in CoinMarketCap table. Columns: {list(cmc_df.columns)}")

    out = cmc_df[[name_col]].copy()
    out.columns = ["exchange_name"]
    out["exchange_name"] = out["exchange_name"].astype(str).str.strip()
    out = out.loc[out["exchange_name"] != ""].head(top_n).reset_index(drop=True)

    return out


def fetch_coingecko_exchange_id_map() -> pd.DataFrame:
    exchange_list = coingecko_get("/exchanges/list")
    exchange_map_df = pd.DataFrame(exchange_list)

    required_cols = ["id", "name"]
    missing_cols = [c for c in required_cols if c not in exchange_map_df.columns]
    if missing_cols:
        raise ValueError(f"CoinGecko exchange list missing columns: {missing_cols}")

    exchange_map_df = exchange_map_df[required_cols].copy()
    exchange_map_df["name_norm"] = (
        exchange_map_df["name"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    return exchange_map_df


def map_cmc_exchanges_to_coingecko_ids(cmc_exchanges_df: pd.DataFrame, coingecko_exchange_map_df: pd.DataFrame) -> pd.DataFrame:
    cmc_df = cmc_exchanges_df.copy()
    cg_df = coingecko_exchange_map_df.copy()

    cmc_df["name_norm"] = cmc_df["exchange_name"].astype(str).str.strip().str.lower()

    mapped_df = cmc_df.merge(
        cg_df[["id", "name", "name_norm"]],
        on="name_norm",
        how="left",
    )

    mapped_df = mapped_df.rename(
        columns={
            "id": "coingecko_exchange_id",
            "name": "coingecko_exchange_name",
        }
    )

    return mapped_df


def fetch_top_exchange_count_for_asset(coingecko_id: str, exchange_ids: list[str]) -> int:
    if not exchange_ids:
        return 0

    page = 1
    matched_exchange_ids = set()

    while True:
        tickers_data = coingecko_get(
            f"/coins/{coingecko_id}/tickers",
            params={
                "exchange_ids": ",".join(exchange_ids),
                "page": page,
            },
        )

        tickers = tickers_data.get("tickers", [])
        if not tickers:
            break

        for ticker in tickers:
            market = ticker.get("market") or {}
            exchange_id = market.get("identifier")
            if exchange_id in exchange_ids:
                matched_exchange_ids.add(exchange_id)

        if len(tickers) < 100:
            break

        page += 1

    return len(matched_exchange_ids)

In [11]:
# 11. Date range
date_list = make_date_list(START_DATE, END_DATE)

print("Date range validated.")
print(f"START_DATE: {START_DATE}")
print(f"END_DATE: {END_DATE}")
print(f"Days to fetch: {len(date_list)}")

Date range validated.
START_DATE: 2026-01-01
END_DATE: 2026-01-31
Days to fetch: 31


In [12]:
# 12. Load existing data and initialize clients
binance = make_binance_exchange()

asset_df = load_existing_assets()
combined_asset_daily_df = load_existing_asset_daily()

history_errors = []
binance_volume_errors = []

appended_asset_daily_total = 0
last_asset_daily_new_df = pd.DataFrame(columns=ASSET_DAILY_COLUMNS)

In [13]:
# 13. Fetch and store asset data
total_asset_steps = len(date_list) * TOP_N
pbar = make_text_progress_bar(total=total_asset_steps, desc="Processing asset data")

for day_idx, run_date in enumerate(date_list, start=1):
    print(f"Day {day_idx}/{len(date_list)} | {run_date}")

    daily_top_assets_df = fetch_top_n_universe_for_date(run_date, TOP_N).copy()
    daily_top_assets_df["date"] = run_date

    daily_universe_df = daily_top_assets_df[
        [
            "date",
            "id",
            "symbol",
            "name",
            "circulating_supply",
            "total_supply",
            "max_supply",
        ]
    ].copy()

    daily_universe_df = daily_universe_df.rename(columns={"id": "coingecko_id"})
    daily_universe_df["symbol"] = daily_universe_df["symbol"].astype(str).str.upper()

    new_asset_core_df = (
        daily_universe_df[
            [
                "coingecko_id",
                "symbol",
                "name",
                "circulating_supply",
                "total_supply",
                "max_supply",
            ]
        ]
        .drop_duplicates(subset=["coingecko_id"], keep="last")
        .reset_index(drop=True)
    )

    asset_df = merge_assets_cumulative(
        existing_assets=asset_df,
        new_asset_core_df=new_asset_core_df,
    )

    asset_lookup_df = (
        daily_universe_df.merge(
            asset_df[["asset_id", "coingecko_id"]],
            on="coingecko_id",
            how="left",
        )[["asset_id", "coingecko_id", "symbol", "name"]]
        .sort_values("asset_id")
        .reset_index(drop=True)
    )

    binance_market_map_df = asset_lookup_df[["asset_id", "symbol"]].drop_duplicates(subset=["asset_id"]).copy()
    binance_market_map_df["market_symbol"] = binance_market_map_df["symbol"].apply(
        lambda s: find_binance_usd_like_symbol(binance, s)
    )

    asset_to_market_symbol = dict(
        zip(binance_market_map_df["asset_id"], binance_market_map_df["market_symbol"])
    )

    history_rows = []
    binance_volume_rows = []

    for _, asset_row in asset_lookup_df.iterrows():
        asset_id = int(asset_row["asset_id"])
        coingecko_id = asset_row["coingecko_id"]
        symbol = asset_row["symbol"]

        try:
            hist_row = fetch_asset_history_row(
                coingecko_id=coingecko_id,
                date_str=run_date,
            )
            hist_row["asset_id"] = asset_id
            history_rows.append(hist_row)
        except Exception as e:
            history_errors.append(
                {
                    "date": run_date,
                    "asset_id": asset_id,
                    "coingecko_id": coingecko_id,
                    "symbol": symbol,
                    "error_message": repr(e),
                }
            )

        market_symbol = asset_to_market_symbol.get(asset_id)
        volume_value = None

        if pd.notna(market_symbol):
            try:
                volume_value = fetch_binance_quote_volume_for_date_with_retry(
                    exchange=binance,
                    market_symbol=str(market_symbol),
                    date_str=run_date,
                )
            except Exception as e:
                binance_volume_errors.append(
                    {
                        "date": run_date,
                        "asset_id": asset_id,
                        "symbol": symbol,
                        "market_symbol": market_symbol,
                        "error_message": repr(e),
                    }
                )

        binance_volume_rows.append(
            {
                "date": run_date,
                "asset_id": asset_id,
                "binance_volume_24h_usd": volume_value,
            }
        )

        pbar.update(1)

    history_df = pd.DataFrame(history_rows)
    binance_volume_df = pd.DataFrame(binance_volume_rows)

    asset_daily_new_df = history_df.merge(
        binance_volume_df[["date", "asset_id", "binance_volume_24h_usd"]],
        on=["date", "asset_id"],
        how="left",
    )

    asset_daily_new_df["binance_volume_24h_usd"] = pd.to_numeric(
        asset_daily_new_df["binance_volume_24h_usd"],
        errors="coerce",
    )

    asset_daily_new_df = asset_daily_new_df[
        [
            "date",
            "asset_id",
            "price_usd",
            "market_cap_usd",
            "global_volume_24h_usd",
            "binance_volume_24h_usd",
        ]
    ].copy()

    asset_daily_new_df = enforce_columns(asset_daily_new_df, ASSET_DAILY_COLUMNS)
    last_asset_daily_new_df = asset_daily_new_df.copy()

    combined_asset_daily_df, appended_asset_daily_df = append_only_new_asset_daily(
        existing_df=combined_asset_daily_df,
        new_df=asset_daily_new_df,
    )
    appended_asset_daily_total += len(appended_asset_daily_df)

    ensure_output_dirs()
    asset_df.to_parquet(ASSETS_PATH, index=False)
    combined_asset_daily_df.to_parquet(ASSET_DAILY_PATH, index=False)

pbar.close()

print("Finished processing all dates.")
print(f"asset rows stored so far: {len(asset_df)}")
print(f"asset_daily rows stored so far: {len(combined_asset_daily_df)}")
print(f"new asset_daily rows appended in this run: {appended_asset_daily_total}")

Processing asset data:   0%|          | 0/3100 [00:00<?, ?it/s]

Day 1/31 | 2026-01-01
Day 2/31 | 2026-01-02
Day 3/31 | 2026-01-03
Day 4/31 | 2026-01-04
Day 5/31 | 2026-01-05
Day 6/31 | 2026-01-06
Day 7/31 | 2026-01-07
Day 8/31 | 2026-01-08
Day 9/31 | 2026-01-09
Day 10/31 | 2026-01-10
Day 11/31 | 2026-01-11
Day 12/31 | 2026-01-12
Day 13/31 | 2026-01-13
Day 14/31 | 2026-01-14
Day 15/31 | 2026-01-15
Day 16/31 | 2026-01-16
Day 17/31 | 2026-01-17
Day 18/31 | 2026-01-18
Day 19/31 | 2026-01-19
Day 20/31 | 2026-01-20
429 for /coins/the-open-network/history | attempt 1/5 | sleeping 5s
429 for /coins/the-open-network/history | attempt 2/5 | sleeping 10s
429 for /coins/the-open-network/history | attempt 3/5 | sleeping 15s
429 for /coins/the-open-network/history | attempt 4/5 | sleeping 20s
429 for /coins/the-open-network/history | attempt 5/5 | sleeping 25s
429 for /coins/tether-gold/history | attempt 1/5 | sleeping 5s
429 for /coins/tether-gold/history | attempt 2/5 | sleeping 10s
429 for /coins/tether-gold/history | attempt 3/5 | sleeping 15s
429 for /coins

KeyboardInterrupt: 

In [ ]:
# 14. Update top-exchange counts
if UPDATE_TOP_EXCHANGE_COUNT:
    cmc_top_exchanges_df = fetch_top_exchange_names_from_cmc(TOP_EXCHANGES_N)
    coingecko_exchange_map_df = fetch_coingecko_exchange_id_map()

    top_exchange_map_df = map_cmc_exchanges_to_coingecko_ids(
        cmc_exchanges_df=cmc_top_exchanges_df,
        coingecko_exchange_map_df=coingecko_exchange_map_df,
    )

    top_exchange_ids = (
        top_exchange_map_df.loc[top_exchange_map_df["coingecko_exchange_id"].notna(), "coingecko_exchange_id"]
        .astype(str)
        .tolist()
    )

    top_exchange_count_rows = []

    pbar = make_text_progress_bar(total=len(asset_df), desc="Updating top-exchange counts")

    for _, asset_row in asset_df.iterrows():
        asset_id = int(asset_row["asset_id"])
        coingecko_id = asset_row["coingecko_id"]

        try:
            top_exchange_count = fetch_top_exchange_count_for_asset(
                coingecko_id=coingecko_id,
                exchange_ids=top_exchange_ids,
            )
        except Exception as e:
            top_exchange_count = pd.NA
            print(f"Top-exchange count fetch failed for {coingecko_id}: {e}")

        top_exchange_count_rows.append(
            {
                "asset_id": asset_id,
                "top_exchange_count": top_exchange_count,
            }
        )

        pbar.update(1)

    pbar.close()

    top_exchange_count_df = pd.DataFrame(top_exchange_count_rows)

    asset_df = asset_df.drop(columns=["top_exchange_count"], errors="ignore").merge(
        top_exchange_count_df,
        on="asset_id",
        how="left",
    )

    asset_df = ensure_asset_columns(asset_df)

    ensure_output_dirs()
    asset_df.to_parquet(ASSETS_PATH, index=False)

else:
    print("Skipping top-exchange count update because UPDATE_TOP_EXCHANGE_COUNT is False.")
    asset_df = ensure_asset_columns(asset_df)

Skipping top-exchange count update.


In [ ]:
# 15. Validation
history_error_df = pd.DataFrame(history_errors)
binance_volume_error_df = pd.DataFrame(binance_volume_errors)

asset_df = ensure_asset_columns(asset_df)
combined_asset_daily_df = enforce_columns(combined_asset_daily_df, ASSET_DAILY_COLUMNS)

if asset_df["asset_id"].duplicated().any():
    raise ValueError("Duplicate asset_id found in asset_df.")

if asset_df["coingecko_id"].duplicated().any():
    raise ValueError("Duplicate coingecko_id found in asset_df.")

if asset_df["symbol"].isna().any() or (asset_df["symbol"].astype(str).str.strip() == "").any():
    raise ValueError("Empty symbol found in asset_df.")

if asset_df["name"].isna().any() or (asset_df["name"].astype(str).str.strip() == "").any():
    raise ValueError("Empty name found in asset_df.")

if last_asset_daily_new_df.duplicated(subset=["date", "asset_id"]).any():
    raise ValueError("Duplicate (date, asset_id) found in newly fetched asset_daily data.")

if combined_asset_daily_df.duplicated(subset=["date", "asset_id"]).any():
    raise ValueError("Duplicate (date, asset_id) found in stored asset_daily data.")

for col in ["market_cap_usd", "global_volume_24h_usd", "binance_volume_24h_usd"]:
    if (pd.to_numeric(combined_asset_daily_df[col], errors="coerce").dropna() < 0).any():
        raise ValueError(f"Negative values found in {col}.")

print("All validation checks passed.")

All validation checks passed.


In [ ]:
# 16. Save final tables
ensure_output_dirs()

asset_df.to_parquet(ASSETS_PATH, index=False)
combined_asset_daily_df.to_parquet(ASSET_DAILY_PATH, index=False)

print(f"Saved asset table: {ASSETS_PATH}")
print(f"Saved asset_daily table: {ASSET_DAILY_PATH}")
print(f"New asset_daily rows appended in this run: {appended_asset_daily_total}")
print(f"Total asset rows stored: {len(asset_df)}")
print(f"Total asset_daily rows stored: {len(combined_asset_daily_df)}")

Saved asset table: ../../data/raw/fetched/assets.parquet
Saved asset_daily table: ../../data/raw/fetched/asset_daily.parquet
New asset_daily rows appended in this run: 4
Total asset rows stored: 101
Total asset_daily rows stored: 4103


In [ ]:
# 17. Output tables and errors
display(asset_df)
display(combined_asset_daily_df.sort_values(["date", "asset_id"]).reset_index(drop=True))

if not history_error_df.empty:
    print("History errors:")
    display(history_error_df)

if not binance_volume_error_df.empty:
    print("Binance volume errors:")
    display(binance_volume_error_df)

,asset_id,coingecko_id,symbol,name,circulating_supply,total_supply,max_supply,top_exchange_count
0,1,bitcoin,BTC,Bitcoin,2.000304e+07,2.000304e+07,2.100000e+07,19
1,2,ethereum,ETH,Ethereum,1.206918e+08,1.206918e+08,NaN,19
2,3,tether,USDT,Tether,1.840921e+11,1.895590e+11,NaN,19
3,4,ripple,XRP,XRP,6.122783e+10,9.998570e+10,1.000000e+11,19
4,5,binancecoin,BNB,BNB,1.363578e+08,1.363578e+08,2.000000e+08,16
...,...,...,...,...,...,...,...,...
96,97,just,JST,JUST,8.815109e+09,8.815109e+09,9.900000e+09,12
97,98,fetch-ai,FET,Artificial Superintelligence Alliance,2.260097e+09,2.714385e+09,2.714385e+09,17
98,99,virtual-protocol,VIRTUAL,Virtuals Protocol,6.563020e+08,1.000000e+09,1.000000e+09,17
99,100,true-usd,TUSD,TrueUSD,4.945151e+08,4.945151e+08,NaN,11


,date,asset_id,price_usd,market_cap_usd,global_volume_24h_usd,binance_volume_24h_usd
0,2026-01-01,1,87520.180487,1.747837e+12,3.724508e+10,5.529916e+08
1,2026-01-01,2,2966.774180,3.580474e+11,1.664149e+10,4.065477e+08
2,2026-01-01,3,0.998663,1.870878e+11,5.819553e+10,8.928731e+05
3,2026-01-01,4,1.839341,1.115841e+11,1.986683e+09,7.621396e+07
4,2026-01-01,5,863.157467,1.188766e+11,1.936923e+09,6.970518e+07
...,...,...,...,...,...,...
4098,2026-03-10,96,0.000006,5.144326e+08,6.233399e+07,5.350813e+06
4099,2026-03-10,97,0.049990,4.406804e+08,1.918851e+07,1.881470e+06
4100,2026-03-10,98,0.143517,3.245597e+08,3.882165e+07,9.390683e+06
4101,2026-03-10,100,0.999655,4.944806e+08,5.724457e+06,2.191258e+05


Missing rows detected in selected date range: 4


,date,asset_id,coingecko_id,symbol,name
0,2026-01-07,101,river,RIVER,River
1,2026-01-08,101,river,RIVER,River
2,2026-01-09,101,river,RIVER,River
3,2026-01-10,101,river,RIVER,River


Unresolved expected rows without asset_id mapping: 0
